# Tesis v5 · B1: CV dentro de cada seed (seeds 1 a 4)
Entorno normal (sin GPU) y **Run all**, en paralelo con los otros dos notebooks.

Reusa de `tesis_backup/v4` los datos sintéticos y la seed 1; guarda en `tesis_backup/v5`. **Solo este notebook (B1)** genera al final la consolidación, el dashboard, el SHAP y las figuras, después de esperar a B2 y B3.

In [ ]:
!pip install -q aif360 catboost lightgbm xgboost shap adjustText


In [ ]:
# ===== CELDA 0 · SETUP (correr primero, en cualquiera de los dos notebooks) =====
import os, json, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
warnings.filterwarnings('ignore')

# --- carpeta de salida en Drive (v5 = CV dentro de cada seed) ---
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/tesis_backup/v5'
V4 = '/content/drive/MyDrive/tesis_backup/v4'   # de acá se reusan los datos sintéticos
os.makedirs(OUT, exist_ok=True)

SEEDS = list(range(1, 11))
target_col = 'default.payment.next.month'
drop_cols = ['ID', 'SEX', target_col, 'AGE_GROUP']

# --- carga y limpieza (idéntico a Bozza tesis) ---
url = "https://raw.githubusercontent.com/YuChenAmberLu/Data-Science--Credit-Card-Default/master/UCI_Credit_Card.csv"
df = pd.read_csv(url)
df_clean = df.copy()
df_clean['EDUCATION'] = df_clean['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df_clean['MARRIAGE'] = df_clean['MARRIAGE'].replace({0: 3})
pay_cols = [c for c in df_clean.columns if c.startswith('PAY_') and 'AMT' not in c]
for c in pay_cols:
    df_clean[c] = df_clean[c].replace({-2: -1, 0: -1})
df_clean['AGE_GROUP'] = df_clean['AGE'].apply(lambda x: 'core' if 25 <= x <= 65 else 'edge')

# --- inyección de sesgo (idéntico a Bozza tesis: una sola vez, seed 42, antes del split) ---
def inject_bias(df, sex_col='SEX', target_col='default.payment.next.month',
                underpriv_code=1, priv_code=2,
                flip_frac_underpriv=0.15, flip_frac_priv=0.15, seed=42):
    rng = np.random.default_rng(seed)
    df2 = df.copy()
    mask_u = (df2[sex_col] == underpriv_code) & (df2[target_col] == 0)
    idx_u = df2[mask_u].index
    flip_idx_u = rng.choice(idx_u, size=int(len(idx_u) * flip_frac_underpriv), replace=False)
    df2.loc[flip_idx_u, target_col] = 1
    mask_p = (df2[sex_col] == priv_code) & (df2[target_col] == 1)
    idx_p = df2[mask_p].index
    flip_idx_p = rng.choice(idx_p, size=int(len(idx_p) * flip_frac_priv), replace=False)
    df2.loc[flip_idx_p, target_col] = 0
    return df2

df_biased = inject_bias(df_clean, underpriv_code=1, priv_code=2)
df_biased['AGE_GROUP'] = pd.factorize(df_biased['AGE_GROUP'])[0]

def fairness_metrics(y_true, y_pred, sensitive, priv_val, underpriv_val):
    def rates(mask):
        yt = np.asarray(y_true)[mask]; yp = np.asarray(y_pred)[mask]
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        ppr = (yp == 1).mean()
        return tpr, fpr, ppr
    sensitive = np.asarray(sensitive)
    tpr_p, fpr_p, ppr_p = rates(sensitive == priv_val)
    tpr_u, fpr_u, ppr_u = rates(sensitive == underpriv_val)
    return {
        'statistical_parity_diff': ppr_p - ppr_u,
        'TPR_priv (women)': tpr_p, 'TPR_underpriv (men)': tpr_u, 'TPR_diff (EOd)': tpr_p - tpr_u,
        'FPR_priv (women)': fpr_p, 'FPR_underpriv (men)': fpr_u, 'FPR_diff (EOd)': fpr_p - fpr_u,
    }

def split(seed):
    """Mismo split que la corrida anterior."""
    return train_test_split(df_biased, test_size=0.2, random_state=seed,
                            stratify=df_biased[target_col])

import uuid
def save_csv_atomic(d, path, **kw):
    """Guarda en un temporal con nombre único y renombra: nunca queda un archivo a medio escribir
    y dos sesiones no pueden pisarse el mismo temporal."""
    tmp = f'{path}.{uuid.uuid4().hex}.tmp'
    d.to_csv(tmp, **kw)
    os.replace(tmp, path)

print("Setup OK", df_biased.shape)
print("Default rate por sexo (original):", df_clean.groupby('SEX')[target_col].mean().round(4).to_dict())
print("Default rate por sexo (con sesgo):", df_biased.groupby('SEX')[target_col].mean().round(4).to_dict())
print("Default rate por edad, original (core = 25-65, edge = resto):",
      df_clean.groupby('AGE_GROUP')[target_col].mean().round(4).to_dict(), "| tamaños:", df_clean['AGE_GROUP'].value_counts().to_dict())


In [ ]:
# ===== CELDA COPIA · Reusar de v4 lo que no cambia (datos sintéticos y la seed 1 completa) =====
import shutil, uuid
reuse = ([f'synth_seed{s}.csv' for s in SEEDS] + [f'test_ids_seed{s}.csv' for s in SEEDS] +
         ['best_params_seed1.json', 'cv_results_seed1.csv', 'results_seed1.csv', 'data_rates_seed1.csv'])
for fn in reuse:
    src_f, dst_f = f'{V4}/{fn}', f'{OUT}/{fn}'
    if os.path.exists(src_f) and not os.path.exists(dst_f):
        tmp = f'{dst_f}.{uuid.uuid4().hex}.tmp'      # nombre único: los 3 notebooks pueden copiar a la vez
        shutil.copyfile(src_f, tmp)
        os.replace(tmp, dst_f)
listos = sum(os.path.exists(f'{OUT}/{fn}') for fn in reuse)
print(f'Archivos de v4 disponibles en v5: {listos} de {len(reuse)}')
assert listos == len(reuse), 'Falta algún archivo en v4: avisá antes de seguir'


In [ ]:
# ===== CELDA B · CV dentro de cada seed + 3 condiciones (CPU alcanza) =====
# Antes: Celda 0. Puede correr en otra sesión mientras la etapa A sigue con el GAN:
# procesa cada seed apenas aparece su synth_seedN.csv en Drive y espera las que faltan.
# Si se corta, volvé a correrla: las seeds completas se saltean y el CV ya hecho se reusa.
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing

# ---------------- configuración ----------------
SEEDS = list(range(1, 11))   # todas las seeds (para la consolidación final)
RUN_SEEDS = [1, 2, 3, 4]          # seeds que procesa ESTE notebook
TUNE_EVERY_SEED = True       # CV dentro de cada seed, sobre su propio train (diseño sin fuga)
N_ITER = 12              # combinaciones probadas por modelo
CV_FOLDS = 3
RW_MODELS = ['LogReg', 'RandomForest', 'GradBoost', 'XGBoost', 'LightGBM', 'CatBoost']

def build_models(seed):
    """Los mismos 9 modelos de antes; los hiperparámetros los pisa el CV."""
    return {
        "LogReg":       make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=seed)),
        "SVM":          make_pipeline(StandardScaler(), LinearSVC(max_iter=5000, random_state=seed)),
        "kNN":          make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
        "MLP":          make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500,
                                                                     early_stopping=True, random_state=seed)),
        "RandomForest": RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1),
        "GradBoost":    GradientBoostingClassifier(random_state=seed),
        "XGBoost":      XGBClassifier(eval_metric="logloss", random_state=seed),
        "LightGBM":     LGBMClassifier(random_state=seed, verbose=-1),
        "CatBoost":     CatBoostClassifier(random_state=seed, verbose=0, allow_writing_files=False),
    }

# espacios de búsqueda (chicos a propósito, para que entre en el día)
SPACES = {
    "LogReg":       {"logisticregression__C": loguniform(1e-3, 1e2)},
    "SVM":          {"linearsvc__C": loguniform(1e-3, 1e1)},
    "kNN":          {"kneighborsclassifier__n_neighbors": randint(5, 101),
                     "kneighborsclassifier__weights": ["uniform", "distance"]},
    "MLP":          {"mlpclassifier__hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64)],
                     "mlpclassifier__alpha": loguniform(1e-5, 1e-1),
                     "mlpclassifier__learning_rate_init": loguniform(3e-4, 3e-3)},
    "RandomForest": {"max_depth": [None, 6, 10, 16],
                     "min_samples_leaf": randint(1, 30),
                     "max_features": ["sqrt", 0.3, 0.5]},
    "GradBoost":    {"n_estimators": randint(100, 301), "learning_rate": loguniform(0.02, 0.3),
                     "max_depth": randint(2, 6), "subsample": uniform(0.7, 0.3)},
    "XGBoost":      {"n_estimators": randint(100, 501), "learning_rate": loguniform(0.02, 0.3),
                     "max_depth": randint(2, 8), "subsample": uniform(0.7, 0.3),
                     "colsample_bytree": uniform(0.6, 0.4), "min_child_weight": randint(1, 11)},
    "LightGBM":     {"n_estimators": randint(100, 501), "learning_rate": loguniform(0.02, 0.3),
                     "num_leaves": randint(8, 64), "min_child_samples": randint(10, 100),
                     "subsample": uniform(0.7, 0.3), "subsample_freq": [1],
                     "colsample_bytree": uniform(0.6, 0.4)},
    "CatBoost":     {"iterations": randint(200, 601), "learning_rate": loguniform(0.02, 0.3),
                     "depth": randint(4, 9), "l2_leaf_reg": loguniform(1, 10)},
}
EXPECTED_ROWS = 2 * len(SPACES) + len(RW_MODELS)   # 9 original + 9 synthetic + 6 reweighed = 24

# ---------------- funciones ----------------
def to_jsonable(params):
    out = {}
    for k, v in params.items():
        if hasattr(v, 'item'):
            v = v.item()
        if isinstance(v, tuple):
            v = list(v)
        out[k] = v
    return out

def from_json(params):
    return {k: (tuple(v) if isinstance(v, list) else v) for k, v in params.items()}

def tune(name, X, y, seed):
    search = RandomizedSearchCV(build_models(seed)[name], SPACES[name], n_iter=N_ITER,
                                scoring='roc_auc',
                                cv=StratifiedKFold(CV_FOLDS, shuffle=True, random_state=seed),
                                random_state=seed, n_jobs=-1, refit=False, return_train_score=True)
    search.fit(X, y)
    return search.best_params_, search.best_score_, pd.DataFrame(search.cv_results_)

def compute_weights(X_tr, y_tr, sex_tr):
    tmp = X_tr.copy()
    tmp['label'] = y_tr.values
    tmp['SEX'] = sex_tr.values
    bld = BinaryLabelDataset(df=tmp, label_names=['label'], protected_attribute_names=['SEX'],
                             favorable_label=0, unfavorable_label=1)
    rw = Reweighing(unprivileged_groups=[{'SEX': 1}], privileged_groups=[{'SEX': 2}])
    return rw.fit_transform(bld).instance_weights

def score_of(model, X):
    """Score continuo para AUC: probabilidad si existe, si no decision_function (LinearSVC)."""
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)

def fit_eval(name, params, seed, X_tr, y_tr, X_te, y_te, sex_te, weights=None):
    model = build_models(seed)[name].set_params(**params)
    if weights is None:
        model.fit(X_tr, y_tr)
    elif hasattr(model, 'steps'):
        model.fit(X_tr, y_tr, **{model.steps[-1][0] + '__sample_weight': weights})
    else:
        model.fit(X_tr, y_tr, sample_weight=weights)
    y_pred = model.predict(X_te)
    m = fairness_metrics(y_te, y_pred, sex_te, priv_val=2, underpriv_val=1)
    m['EOd_max'] = max(abs(m['TPR_diff (EOd)']), abs(m['FPR_diff (EOd)']))
    m['accuracy'] = accuracy_score(y_te, y_pred)
    m['balanced_accuracy'] = balanced_accuracy_score(y_te, y_pred)
    m['roc_auc'] = roc_auc_score(y_te, score_of(model, X_te))
    m['pred_positive_rate'] = float(np.mean(y_pred))   # detecta modelos que predicen casi todo una clase
    return m, model, y_pred

def get_params(seed, X_tr, y_tr):
    """Reusa el CV si ya existe (cortes) o el de la seed 1 si TUNE_EVERY_SEED=False; si no, lo corre."""
    own = f'{OUT}/best_params_seed{seed}.json'
    shared = f'{OUT}/best_params_seed{SEEDS[0]}.json'
    for p in ([own] if TUNE_EVERY_SEED else [own, shared]):
        if os.path.exists(p):
            with open(p) as fh:
                print(f'  hiperparámetros reusados de {os.path.basename(p)}')
                return {k: from_json(v) for k, v in json.load(fh).items()}
    best, cv_all, t_cv = {}, [], time.time()
    for name in SPACES:
        t1 = time.time()
        bp, bs, cvr = tune(name, X_tr, y_tr, seed)
        best[name] = bp
        cvr['model'] = name
        cvr['seed'] = seed
        cv_all.append(cvr)
        print(f'  CV {name:12s} AUC={bs:.4f}  ({time.time() - t1:.0f}s)  {to_jsonable(bp)}')
    print(f'  CV total: {(time.time() - t_cv) / 60:.1f} min')
    save_csv_atomic(pd.concat(cv_all, ignore_index=True), f'{OUT}/cv_results_seed{seed}.csv', index=False)
    tmp = f'{own}.{uuid.uuid4().hex}.tmp'
    with open(tmp, 'w') as fh:
        json.dump({k: to_jsonable(v) for k, v in best.items()}, fh, indent=1)
    os.replace(tmp, own)
    return best

def group_rates(y, sex, w=None):
    """Tasa de default por sexo (ponderada si hay pesos) y brecha mujeres − hombres."""
    w = np.ones(len(y)) if w is None else np.asarray(w)
    y, sex = np.asarray(y), np.asarray(sex)
    r = {g: float(np.sum(w[sex == g] * y[sex == g]) / np.sum(w[sex == g])) for g in (1, 2)}
    return {'default_rate_men': r[1], 'default_rate_women': r[2], 'gap_women_minus_men': r[2] - r[1]}

def seed_done(seed):
    p = f'{OUT}/results_seed{seed}.csv'
    if not os.path.exists(p):
        return False
    r = pd.read_csv(p)
    return len(r) == EXPECTED_ROWS and {'roc_auc', 'balanced_accuracy'} <= set(r.columns)

def run_seed(seed):
    t0 = time.time()
    df_tr, df_te = split(seed)
    ids_path = f'{OUT}/test_ids_seed{seed}.csv'
    if os.path.exists(ids_path):
        assert (pd.read_csv(ids_path)['ID'].values == df_te['ID'].values).all(), \
            f'seed {seed}: el split no coincide con el de la etapa A'

    X_tr, y_tr, sex_tr = df_tr.drop(columns=drop_cols), df_tr[target_col], df_tr['SEX']
    X_te, y_te, sex_te = df_te.drop(columns=drop_cols), df_te[target_col], df_te['SEX']
    synth = pd.read_csv(f'{OUT}/synth_seed{seed}.csv')
    X_s = synth[X_tr.columns]                 # mismas columnas y orden que los datos reales
    y_s = synth['default'].astype(int)
    weights = compute_weights(X_tr, y_tr, sex_tr)

    # datos de entrenamiento de cada condición: tasa de default por sexo (ejemplo para el texto
    # y control de que FairFinGAN no invierta la desigualdad)
    rates = pd.DataFrame([
        {'seed': seed, 'data': 'original_train', **group_rates(y_tr, sex_tr)},
        {'seed': seed, 'data': 'reweighed_train', **group_rates(y_tr, sex_tr, weights)},
        {'seed': seed, 'data': 'synthetic_train', **group_rates(y_s, synth['SEX'])},
    ])
    save_csv_atomic(rates, f'{OUT}/data_rates_seed{seed}.csv', index=False)
    print(rates[['data', 'default_rate_men', 'default_rate_women', 'gap_women_minus_men']].round(4).to_string(index=False))

    params = get_params(seed, X_tr, y_tr)

    rows = []
    for name in SPACES:
        p = params[name]
        m, _, _ = fit_eval(name, p, seed, X_tr, y_tr, X_te, y_te, sex_te)
        rows.append({'seed': seed, 'dataset': 'original', 'model': name, **m})
        m, _, _ = fit_eval(name, p, seed, X_s, y_s, X_te, y_te, sex_te)
        rows.append({'seed': seed, 'dataset': 'synthetic', 'model': name, **m})
        if name in RW_MODELS:
            m, _, _ = fit_eval(name, p, seed, X_tr, y_tr, X_te, y_te, sex_te, weights=weights)
            rows.append({'seed': seed, 'dataset': 'reweighed', 'model': name, **m})

    res = pd.DataFrame(rows)
    assert len(res) == EXPECTED_ROWS
    save_csv_atomic(res, f'{OUT}/results_seed{seed}.csv', index=False)
    print(f'===== seed {seed} lista en {(time.time() - t0) / 60:.1f} min =====')

# ---------------- loop: procesa las seeds a medida que la etapa A las termina ----------------
pending = [s for s in RUN_SEEDS if not seed_done(s)]
while pending:
    ready = [s for s in pending if os.path.exists(f'{OUT}/synth_seed{s}.csv')]
    if not ready:
        print(time.strftime('%H:%M'), '· esperando la etapa A para las seeds', pending)
        time.sleep(120)
        continue
    seed = ready[0]
    print(f'===== seed {seed} =====')
    run_seed(seed)
    pending.remove(seed)
print("Etapa B terminada para las seeds", RUN_SEEDS)


In [ ]:
# ===== CELDA C · Consolidar, resumir y comparar con la corrida anterior (con las 10 seeds listas) =====
missing = [s for s in SEEDS if not seed_done(s)]
while missing:   # espera a que los otros notebooks terminen sus seeds
    print(time.strftime('%H:%M'), '· esperando las seeds de los otros notebooks:', missing)
    time.sleep(120)
    missing = [s for s in SEEDS if not seed_done(s)]
new = pd.concat([pd.read_csv(f'{OUT}/results_seed{s}.csv') for s in SEEDS], ignore_index=True)
for c in ['statistical_parity_diff', 'TPR_diff (EOd)', 'FPR_diff (EOd)']:
    new['abs_' + c] = new[c].abs()          # magnitudes: +0.20 y −0.20 no se cancelan
save_csv_atomic(new, f'{OUT}/all_conditions_v5.csv', index=False)

# --- resumen por modelo y condición: media, desvío y error estándar ---
cols = ['abs_statistical_parity_diff', 'abs_TPR_diff (EOd)', 'abs_FPR_diff (EOd)', 'EOd_max',
        'statistical_parity_diff', 'TPR_diff (EOd)', 'FPR_diff (EOd)',
        'accuracy', 'balanced_accuracy', 'roc_auc', 'pred_positive_rate']
summary = new.groupby(['model', 'dataset'])[cols].agg(['mean', 'std', 'sem'])
summary.to_csv(f'{OUT}/summary_mean_sd_se_v5.csv')

# --- diferencias emparejadas por seed vs original ---
# se_naive = sd/sqrt(k), el criterio del draft. se_nb = corrección de Nadeau & Bengio (2003) para
# splits repetidos que comparten observaciones: sd*sqrt(1/k + n_test/n_train). Es más conservadora;
# los dos son indicadores descriptivos, no un test formal.
TEST_TRAIN_RATIO = 0.2 / 0.8

def paired_deltas(d, metrics):
    out = []
    for (model, cond), g in d[d.dataset != 'original'].groupby(['model', 'dataset']):
        o = d[(d.model == model) & (d.dataset == 'original')].set_index('seed')
        g = g.set_index('seed')
        idx = g.index.intersection(o.index)
        k = len(idx)
        for metric in metrics:
            if metric not in g.columns:
                continue
            if metric == 'statistical_parity_diff':
                delta = g.loc[idx, metric].abs() - o.loc[idx, metric].abs()
            else:
                delta = g.loc[idx, metric] - o.loc[idx, metric]
            mean, sd = delta.mean(), delta.std(ddof=1)
            se_naive, se_nb = sd / np.sqrt(k), sd * np.sqrt(1 / k + TEST_TRAIN_RATIO)
            out.append({'model': model, 'cond': cond, 'metric': metric, 'k': k, 'mean': mean, 'sd': sd,
                        'se_naive': se_naive, 'se_nb': se_nb,
                        'consistent_2se': abs(mean) > 2 * se_naive,
                        'consistent_2se_nb': abs(mean) > 2 * se_nb})
    return pd.DataFrame(out)

M_NEW = ['statistical_parity_diff', 'EOd_max', 'accuracy', 'balanced_accuracy', 'roc_auc']
deltas_new = paired_deltas(new, M_NEW)
save_csv_atomic(deltas_new, f'{OUT}/paired_deltas_v5.csv', index=False)
print("Resultados consistentes (2·SE del draft):", int(deltas_new.consistent_2se.sum()),
      "| con corrección Nadeau-Bengio:", int(deltas_new.consistent_2se_nb.sum()), "de", len(deltas_new))

# --- comparación con la corrida anterior (solo métricas que existían antes) ---
OLD_PATH = '/content/drive/MyDrive/tesis_backup/all_conditions_final.csv'
M_OLD = ['statistical_parity_diff', 'EOd_max', 'accuracy']
if not os.path.exists(OLD_PATH):
    print("\nNo encuentro la corrida anterior en", OLD_PATH, "→ salteo la comparación.")
else:
    old = pd.read_csv(OLD_PATH)
    need = {'seed', 'dataset', 'model'} | set(M_OLD)
    falta = need - set(old.columns)
    assert not falta, f'Al CSV viejo le faltan columnas: {falta}'
    cmp = paired_deltas(old, M_OLD).merge(paired_deltas(new, M_OLD), on=['model', 'cond', 'metric'],
                                          suffixes=('_old', '_new'))
    cmp['cambia_signo'] = np.sign(cmp['mean_old']) != np.sign(cmp['mean_new'])
    cmp['cambia_consistencia'] = cmp['consistent_2se_old'] != cmp['consistent_2se_new']
    save_csv_atomic(cmp, f'{OUT}/comparacion_old_vs_new.csv', index=False)
    chg = cmp[cmp.cambia_signo | cmp.cambia_consistencia]
    print(f"\nComparación con la corrida anterior: {len(cmp)} resultados, {len(chg)} cambian (signo o consistencia):")
    show = ['model', 'cond', 'metric', 'mean_old', 'consistent_2se_old', 'mean_new', 'consistent_2se_new']
    print(chg[show].round(4).to_string(index=False) if len(chg) else "  ninguno: las conclusiones del draft se mantienen")

# --- comparación v4 (CV solo en la seed 1) vs v5 (CV en cada seed) ---
V4_ALL = f'{V4}/all_conditions_v4.csv'
if os.path.exists(V4_ALL):
    v4 = pd.read_csv(V4_ALL)
    cmp45 = paired_deltas(v4, M_NEW).merge(deltas_new, on=['model', 'cond', 'metric'], suffixes=('_v4', '_v5'))
    cmp45['cambia_signo'] = np.sign(cmp45['mean_v4']) != np.sign(cmp45['mean_v5'])
    cmp45['cambia_consistencia_nb'] = cmp45['consistent_2se_nb_v4'] != cmp45['consistent_2se_nb_v5']
    save_csv_atomic(cmp45, f'{OUT}/comparacion_v4_vs_v5.csv', index=False)
    ch = cmp45[cmp45.cambia_signo | cmp45.cambia_consistencia_nb]
    print(f"\nv4 vs v5: {len(cmp45)} resultados, {len(ch)} cambian de signo o de consistencia (criterio corregido)")
    print(ch[['model', 'cond', 'metric', 'mean_v4', 'mean_v5', 'consistent_2se_nb_v4', 'consistent_2se_nb_v5']].round(4).to_string(index=False) if len(ch) else "  ninguno")

# --- tasas de default por sexo en los datos de cada condición (ejemplo para el texto) ---
rates = pd.concat([pd.read_csv(f'{OUT}/data_rates_seed{s}.csv') for s in SEEDS], ignore_index=True)
save_csv_atomic(rates, f'{OUT}/data_rates_all.csv', index=False)
print(f"\nTasa de default por sexo en los datos de entrenamiento (media de {len(SEEDS)} seeds):")
print(rates.groupby('data')[['default_rate_men', 'default_rate_women', 'gap_women_minus_men']].agg(['mean', 'std']).round(4).to_string())

# --- TPR y FPR de cada grupo junto a las brechas (para ver si una brecha baja porque empeoran ambos) ---
grp_cols = ['TPR_underpriv (men)', 'TPR_priv (women)', 'FPR_underpriv (men)', 'FPR_priv (women)', 'pred_positive_rate']
by_group = new.groupby(['model', 'dataset'])[grp_cols].agg(['mean', 'std'])
by_group.to_csv(f'{OUT}/group_rates_v5.csv')
print("\nGuardado group_rates_v5.csv (TPR y FPR por grupo)")


In [ ]:
# ===== CELDA D · Predicciones para el dashboard + SHAP (seed 1, separado del experimento) =====
# Se puede correr apenas la seed 1 esté lista en la etapa B. Si SHAP falla, no afecta los resultados.
MONITOR_SEED = 1
with open(f'{OUT}/best_params_seed{MONITOR_SEED}.json') as fh:
    params = {k: from_json(v) for k, v in json.load(fh).items()}

df_tr, df_te = split(MONITOR_SEED)
X_tr, y_tr = df_tr.drop(columns=drop_cols), df_tr[target_col]
X_te, y_te, sex_te = df_te.drop(columns=drop_cols), df_te[target_col], df_te['SEX']
synth = pd.read_csv(f'{OUT}/synth_seed{MONITOR_SEED}.csv')
X_s, y_s = synth[X_tr.columns], synth['default'].astype(int)

preds, xgb_original = [], None
for name in SPACES:
    for ds, (Xa, ya) in {'original': (X_tr, y_tr), 'synthetic': (X_s, y_s)}.items():
        _, model, yp = fit_eval(name, params[name], MONITOR_SEED, Xa, ya, X_te, y_te, sex_te)
        preds.append(pd.DataFrame({'seed': MONITOR_SEED, 'obs_id': df_te['ID'].values, 'model': name,
                                   'dataset': ds, 'y_true': y_te.values.astype(int),
                                   'y_pred': np.asarray(yp).astype(int), 'sex': sex_te.values.astype(int),
                                   'age_group': df_te['AGE_GROUP'].values.astype(int)}))
        if name == 'XGBoost' and ds == 'original':
            xgb_original = model
pred_df = pd.concat(preds, ignore_index=True)
save_csv_atomic(pred_df, f'{OUT}/monitor_predictions_seed{MONITOR_SEED}_v5.csv', index=False)
print("Predicciones del dashboard guardadas:", pred_df.shape)

try:
    import shap
    sv = shap.TreeExplainer(xgb_original).shap_values(X_te)
    imp = pd.DataFrame({'feature': X_te.columns, 'mean_abs_shap': np.abs(sv).mean(axis=0)})
    imp['corr_with_SEX'] = imp['feature'].map(X_te.corrwith(sex_te.astype(float)).abs())
    age_bin = X_te['AGE'].between(25, 65).astype(float)          # atributo secundario (core vs edge)
    imp['corr_with_AGE_GROUP'] = imp['feature'].map(X_te.corrwith(age_bin).abs())
    imp = imp.sort_values('mean_abs_shap', ascending=False)
    save_csv_atomic(imp, f'{OUT}/shap_importance_v5.csv', index=False)
    print(imp.head(10).round(4).to_string(index=False))
    import matplotlib.pyplot as plt
    top = imp.head(15).iloc[::-1]
    plt.figure(figsize=(8, 6))
    plt.barh(top['feature'], top['mean_abs_shap'], color='#8aa6c1')
    plt.xlabel('Mean |SHAP value|')
    plt.title('XGBoost (tuned, seed 1): feature importance', fontweight='bold')
    plt.tight_layout(); plt.savefig(f'{OUT}/fig_shap.png', dpi=200, bbox_inches='tight'); plt.show()
except Exception as e:
    print("SHAP falló (los resultados del experimento no se ven afectados):", repr(e))


In [ ]:
# ===== CELDA E · Gráficos del cross validation (los que pidió Carman) =====
import matplotlib.pyplot as plt

cv = pd.concat([pd.read_csv(f'{OUT}/cv_results_seed{s}.csv') for s in SEEDS
                if os.path.exists(f'{OUT}/cv_results_seed{s}.csv')], ignore_index=True)
print("Seeds con CV:", sorted(cv.seed.unique()))

# hiperparámetro principal de cada modelo para el eje x (log = escala logarítmica)
MAIN = {'LogReg': ('param_logisticregression__C', 'C', True),
        'SVM': ('param_linearsvc__C', 'C', True),
        'kNN': ('param_kneighborsclassifier__n_neighbors', 'k neighbours', False),
        'MLP': ('param_mlpclassifier__alpha', 'alpha (L2)', True),
        'RandomForest': ('param_min_samples_leaf', 'min_samples_leaf', False),
        'GradBoost': ('param_learning_rate', 'learning_rate', True),
        'XGBoost': ('param_learning_rate', 'learning_rate', True),
        'LightGBM': ('param_num_leaves', 'num_leaves', False),
        'CatBoost': ('param_depth', 'depth', False)}

# 1) Resultados de la búsqueda aleatoria (exploratorio): AUC vs hiperparámetro principal.
#    No son curvas de validación en sentido estricto: los demás hiperparámetros también varían.
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, (model, (col, label, logx)) in zip(axes.ravel(), MAIN.items()):
    d = cv[cv.model == model].copy()
    d[col] = pd.to_numeric(d[col], errors='coerce')
    d = d.dropna(subset=[col]).sort_values(col)
    ax.scatter(d[col], d['mean_train_score'], s=14, color='#bbbbbb', label='training')
    ax.errorbar(d[col], d['mean_test_score'], yerr=d['std_test_score'], fmt='o', ms=4,
                color='#c1664f', ecolor='#e2b3a6', capsize=2, label='validation')
    best = d[d.rank_test_score == 1]
    ax.scatter(best[col], best['mean_test_score'], s=70, facecolors='none', edgecolors='black',
               label='selected (per seed)')
    if logx:
        ax.set_xscale('log')
    ax.set_title(model, fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('ROC-AUC')
    ax.grid(True, alpha=0.2)
axes[0, 0].legend(fontsize=8)
fig.suptitle('Random search (exploratory): cross-validated ROC-AUC vs main hyperparameter, all seeds', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT}/fig_cv_search.png', dpi=200, bbox_inches='tight')
plt.show()

# 2) Entrenamiento vs validación del modelo elegido (sobreajuste), media ± SE entre seeds
b = cv[cv.rank_test_score == 1].groupby(['model', 'seed'])[['mean_train_score', 'mean_test_score']].first()
agg = b.groupby('model').agg(['mean', 'sem']).reindex(list(MAIN))
x = np.arange(len(agg))
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - 0.2, agg[('mean_train_score', 'mean')], 0.4, yerr=agg[('mean_train_score', 'sem')],
       capsize=3, color='#bbbbbb', label='training')
ax.bar(x + 0.2, agg[('mean_test_score', 'mean')], 0.4, yerr=agg[('mean_test_score', 'sem')],
       capsize=3, color='#c1664f', label='validation')
ax.set_xticks(x)
ax.set_xticklabels(agg.index, rotation=30, ha='right')
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('ROC-AUC')
ax.set_title('Selected configuration: training vs validation AUC (mean ± SE across seeds)', fontweight='bold')
ax.legend()
ax.grid(True, axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_cv_train_vs_val.png', dpi=200, bbox_inches='tight')
plt.show()

# 3) Tabla de hiperparámetros elegidos por seed (estabilidad de la selección)
rows = []
for s in SEEDS:
    p = f'{OUT}/best_params_seed{s}.json'
    if os.path.exists(p):
        with open(p) as fh:
            for model, prm in json.load(fh).items():
                rows.append({'seed': s, 'model': model, **{k.split('__')[-1]: v for k, v in prm.items()}})
best_all = pd.DataFrame(rows)
save_csv_atomic(best_all, f'{OUT}/best_params_all_seeds.csv', index=False)
print("Guardados: fig_cv_search.png, fig_cv_train_vs_val.png, best_params_all_seeds.csv")


In [ ]:
# ===== CELDA F · Figuras de resultados de la tesis (las de Bozza tesis, con los datos nuevos) =====
# heatmap, forest plot, barras y trade-off. Lee all_conditions_v5.csv (celda C).
import matplotlib.pyplot as plt
import seaborn as sns

res = pd.read_csv(f'{OUT}/all_conditions_v5.csv')
order = ['LogReg', 'SVM', 'kNN', 'MLP', 'RandomForest', 'GradBoost', 'XGBoost', 'LightGBM', 'CatBoost']
COL = {'original': '#8aa6c1', 'synthetic': '#c1664f', 'reweighed': '#6a9a6a'}
NAME = {'original': 'Original', 'synthetic': 'FairFinGAN', 'reweighed': 'Reweighing'}

def paired(cond, metric):
    """Δ por seed vs original (|SP| o EOd_max): media y SE corregido (Nadeau-Bengio)."""
    out = []
    for m in order:
        o = res[(res.model == m) & (res.dataset == 'original')].set_index('seed')[metric]
        s = res[(res.model == m) & (res.dataset == cond)].set_index('seed')[metric]
        idx = s.index.intersection(o.index)
        if len(idx) == 0:
            out.append((m, np.nan, np.nan)); continue
        dlt = s.loc[idx].abs() - o.loc[idx].abs()
        sd = dlt.std(ddof=1) if len(idx) > 1 else np.nan
        out.append((m, dlt.mean(), sd * np.sqrt(1 / len(idx) + 0.2 / 0.8)))
    return out

# 1) Heatmap: efecto de cada mitigación por modelo y métrica
heat = pd.DataFrame({
    'FairFinGAN\nSP':  [d for _, d, _ in paired('synthetic', 'statistical_parity_diff')],
    'FairFinGAN\nEOd': [d for _, d, _ in paired('synthetic', 'EOd_max')],
    'Reweighing\nSP':  [d for _, d, _ in paired('reweighed', 'statistical_parity_diff')],
    'Reweighing\nEOd': [d for _, d, _ in paired('reweighed', 'EOd_max')],
}, index=order)
plt.figure(figsize=(8, 7))
sns.heatmap(heat, annot=True, fmt='+.3f', cmap='RdBu_r', center=0, mask=heat.isna(),
            cbar_kws={'label': 'Δ fairness gap vs original (negative = fairer)'},
            linewidths=0.5, linecolor='white')
plt.title('Mitigation effect by model and metric', fontweight='bold', pad=12)
plt.ylabel('Model'); plt.xlabel('')
plt.tight_layout(); plt.savefig(f'{OUT}/fig_heatmap.png', dpi=200, bbox_inches='tight'); plt.show()

# 2) Forest plot: Δ media ± 2·SE (corregido) por modelo
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
for ax, metric, label in zip(axes, ['statistical_parity_diff', 'EOd_max'], ['Statistical parity', 'Equalized odds']):
    for cond, off, mk in [('synthetic', 0.15, 'o'), ('reweighed', -0.15, 's')]:
        for i, (m, mean, se) in enumerate(paired(cond, metric)):
            if not np.isnan(mean):
                ax.errorbar(mean, i + off, xerr=2 * se, fmt=mk, color=COL[cond], capsize=3)
        ax.errorbar([], [], fmt=mk, color=COL[cond], label=NAME[cond])
    ax.axvline(0, color='gray', lw=1, ls='--')
    ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Δ vs original, mean ± 2·SE, Nadeau-Bengio corrected  (← fairer)')
axes[0].invert_yaxis(); axes[0].legend(loc='lower right')
plt.tight_layout(); plt.savefig(f'{OUT}/fig_forest.png', dpi=200, bbox_inches='tight'); plt.show()

# 3) Barras: |SP|, |brecha TPR| y |brecha FPR| por condición
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (metric, label) in zip(axes, [('statistical_parity_diff', 'Statistical parity'),
                                       ('TPR_diff (EOd)', 'TPR gap'), ('FPR_diff (EOd)', 'FPR gap')]):
    x, w = np.arange(len(order)), 0.27
    for k, cond in enumerate(['original', 'synthetic', 'reweighed']):
        vals = [res[(res.model == m) & (res.dataset == cond)][metric].abs() for m in order]
        ax.bar(x + (k - 1) * w, [v.mean() if len(v) else np.nan for v in vals], w,
               yerr=[v.sem() if len(v) > 1 else 0 for v in vals], capsize=2,
               color=COL[cond], label=NAME[cond])
    ax.set_xticks(x); ax.set_xticklabels(order, rotation=45, ha='right')
    ax.set_title(f'|{label}|  (closer to 0 = fairer; bars: ± SE, uncorrected)', fontweight='bold', fontsize=10)
    ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_bars.png', dpi=200, bbox_inches='tight'); plt.show()

# 4) Trade-off fairness vs rendimiento (|SP| vs accuracy y vs AUC)
try:
    from adjustText import adjust_text
except ImportError:
    adjust_text = None
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, perf, plabel in zip(axes, ['accuracy', 'roc_auc'], ['Accuracy', 'ROC-AUC']):
    texts = []
    for cond in ['original', 'synthetic', 'reweighed']:
        for m in order:
            d = res[(res.model == m) & (res.dataset == cond)]
            if len(d) == 0:
                continue
            xv, yv = d['statistical_parity_diff'].abs().mean(), d[perf].mean()
            ax.scatter(xv, yv, color=COL[cond], s=60, zorder=3, edgecolor='white', linewidth=0.8)
            texts.append(ax.text(xv, yv, m, fontsize=7, color=COL[cond]))
    for cond in COL:
        ax.scatter([], [], color=COL[cond], label=NAME[cond], s=60)
    if adjust_text:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='lightgray', lw=0.5))
    ax.set_xlabel('|Statistical parity|  (← fairer)'); ax.set_ylabel(f'{plabel}  (higher = better)')
    ax.set_title(f'Fairness–{plabel.lower()} trade-off', fontweight='bold')
    ax.legend(loc='best'); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_tradeoff.png', dpi=200, bbox_inches='tight'); plt.show()
print("Guardadas: fig_heatmap.png, fig_forest.png, fig_bars.png, fig_tradeoff.png")
